# Shared activation trajectory model across line lengths

This notebook runs the first trajectory-generalization experiment:

1. Collect pre-final-attention activations from training and held-out examples.
2. Fit one global 3D PCA using training activations only.
3. Represent each position by normalized within-line phase `u = character_count / line_length`.
4. Fit a shared smooth spline trajectory with a line-length interaction, plus token-position and boundary-residual terms.
5. Evaluate held-out reconstruction error and inspect residuals by phase.
6. Display a randomly selected held-out example in the global PCA space, with actual and predicted trajectories.

The model is fit to the three global PCA coordinates. The held-out metrics therefore measure how well this low-dimensional, phase-aligned trajectory model predicts the activation geometry.

In [7]:
from pathlib import Path
import json

import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import SplineTransformer

import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

RUN_DIR = Path("../results/line_breaks_results")
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "final.pt"
N_TRAIN_EXAMPLES = 2_000
N_EVAL_EXAMPLES = 500
COLLECTION_BATCH_SIZE = 128
TRAIN_SEED = 22_001
EVAL_SEED = 22_002
N_SPLINE_KNOTS = 12
RIDGE_ALPHA = 1.0
RANDOM_EXAMPLE_SEED = 22_003
INCLUDE_TOKEN_POSITION = True


In [8]:
with open(RUN_DIR / "config.json", "r") as f:
    cfg = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
vocab = project_tokens.build_vocab(cfg["task"])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(path=CHECKPOINT_PATH, model=model, map_location=device)
model.eval()

n_layers = cfg["model"]["n_layers"]
final_layer = n_layers - 1
ACTIVATION_SITE = f"blocks.{final_layer}.hook_resid_pre"
seq_len = cfg["task"]["seq_len"]
print(f"Loaded checkpoint step {checkpoint['step']} on {device}")
print(f"Activation site: {ACTIVATION_SITE}")
print(f"Train/eval examples: {N_TRAIN_EXAMPLES:,}/{N_EVAL_EXAMPLES:,}")


Building vocab...
('BOS', '_NEWLINE_')
Loaded checkpoint step 10000 on cpu
Activation site: blocks.2.hook_resid_pre
Train/eval examples: 2,000/500


## Collect activation trajectories

In [9]:
def character_counts_for_tokens(token_rows):
    all_counts = []
    for row in token_rows.detach().cpu().tolist():
        running_count = 0
        row_counts = []
        for token_id in row:
            if token_id == vocab.newline_id:
                running_count = 0
            else:
                running_count += project_data.extract_character_count(vocab.decode_token(token_id))
            row_counts.append(running_count)
        all_counts.append(row_counts)
    return torch.tensor(all_counts, dtype=torch.float32)

@torch.no_grad()
def collect_examples(n_examples, seed):
    activation_chunks, count_chunks, width_chunks, token_chunks = [], [], [], []
    collected = 0
    batch_number = 0
    while collected < n_examples:
        batch = project_data.make_batch(
            batch_size=min(COLLECTION_BATCH_SIZE, n_examples - collected),
            vocab=vocab, task_cfg=cfg["task"], device=device,
            seed=seed + batch_number * COLLECTION_BATCH_SIZE,
        )
        _, cache = model.run_with_cache(
            batch.tokens, prepend_bos=False,
            names_filter=lambda name: name == ACTIVATION_SITE,
        )
        activation_chunks.append(cache[ACTIVATION_SITE].detach().cpu())
        count_chunks.append(character_counts_for_tokens(batch.tokens))
        width_chunks.append(batch.line_lengths.detach().cpu().float())
        token_chunks.append(batch.tokens.detach().cpu())
        collected += len(batch.tokens)
        batch_number += 1
    return (
        torch.cat(activation_chunks, dim=0)[:n_examples].numpy(),
        torch.cat(count_chunks, dim=0)[:n_examples].numpy(),
        torch.cat(width_chunks, dim=0)[:n_examples].numpy(),
        torch.cat(token_chunks, dim=0)[:n_examples].numpy(),
    )

X_train, counts_train, widths_train, tokens_train = collect_examples(N_TRAIN_EXAMPLES, TRAIN_SEED)
X_eval, counts_eval, widths_eval, tokens_eval = collect_examples(N_EVAL_EXAMPLES, EVAL_SEED)
print("Train activations:", X_train.shape)
print("Eval activations: ", X_eval.shape)


Train activations: (2000, 150, 128)
Eval activations:  (500, 150, 128)


## Fit one global PCA on training activations

In [10]:
d_model = X_train.shape[-1]
pca = PCA(n_components=3, svd_solver="randomized", random_state=RANDOM_EXAMPLE_SEED)
pca.fit(X_train.reshape(-1, d_model))
Y_train = pca.transform(X_train.reshape(-1, d_model)).reshape(N_TRAIN_EXAMPLES, seq_len, 3)
Y_eval = pca.transform(X_eval.reshape(-1, d_model)).reshape(N_EVAL_EXAMPLES, seq_len, 3)
print(f"Global PCA variance explained: {pca.explained_variance_ratio_.sum():.1%}")
print("Per-component variance explained:", pca.explained_variance_ratio_)


Global PCA variance explained: 88.3%
Per-component variance explained: [0.5800768  0.20581079 0.09695096]


## Fit a shared phase spline with line-length interaction

In [11]:
def make_features(character_counts, line_lengths):
    n_examples, n_positions = character_counts.shape
    phase = character_counts / line_lengths[:, None]
    phase = np.clip(phase, 0.0, 1.5)
    line_length_centered = (
        line_lengths - widths_train.mean()
    ) / max(widths_train.std(), 1.0)
    boundary_residual = (character_counts - line_lengths[:, None]) / max(widths_train.mean(), 1.0)
    token_position = np.broadcast_to(
        np.arange(n_positions, dtype=np.float32) / max(n_positions - 1, 1),
        (n_examples, n_positions),
    )

    flat_phase = phase.reshape(-1, 1)
    spline = phase_spline.transform(flat_phase)
    line_feature = np.repeat(line_length_centered, n_positions)[:, None]
    features = [spline, spline * line_feature, boundary_residual.reshape(-1, 1)]
    if INCLUDE_TOKEN_POSITION:
        features.append(token_position.reshape(-1, 1))
    return np.concatenate(features, axis=1)

phase_spline = SplineTransformer(
    n_knots=N_SPLINE_KNOTS, degree=3, include_bias=False,
)
phase_spline.fit(np.linspace(0.0, 1.5, 500)[:, None])
F_train = make_features(counts_train, widths_train)
F_eval = make_features(counts_eval, widths_eval)

trajectory_model = Ridge(alpha=RIDGE_ALPHA)
trajectory_model.fit(F_train, Y_train.reshape(-1, 3))
Y_train_pred = trajectory_model.predict(F_train).reshape(N_TRAIN_EXAMPLES, seq_len, 3)
Y_eval_pred = trajectory_model.predict(F_eval).reshape(N_EVAL_EXAMPLES, seq_len, 3)
print("Feature matrix:", F_train.shape)


Feature matrix: (300000, 28)


## Held-out evaluation

In [12]:
def summarize_fit(name, actual, predicted):
    actual_flat = actual.reshape(-1, 3)
    predicted_flat = predicted.reshape(-1, 3)
    mse = float(np.mean((actual_flat - predicted_flat) ** 2))
    r2 = float(r2_score(actual_flat, predicted_flat, multioutput="variance_weighted"))
    per_component_r2 = r2_score(actual_flat, predicted_flat, multioutput="raw_values")
    print(f"{name}: MSE={mse:.6f}; weighted R²={r2:.4f}; per-PC R²={per_component_r2}")
    return mse, r2

summarize_fit("Train", Y_train, Y_train_pred)
summarize_fit("Held-out", Y_eval, Y_eval_pred)

# Baselines: global mean and phase-only spline without line-length interaction.
global_mean = Y_train.mean(axis=(0, 1))
baseline_pred = np.broadcast_to(global_mean, Y_eval.shape)
summarize_fit("Held-out global-mean baseline", Y_eval, baseline_pred)

phase_only_model = Ridge(alpha=RIDGE_ALPHA)
phase_only_model.fit(F_train[:, :phase_spline.n_features_out_], Y_train.reshape(-1, 3))
phase_only_pred = phase_only_model.predict(F_eval[:, :phase_spline.n_features_out_]).reshape(N_EVAL_EXAMPLES, seq_len, 3)
summarize_fit("Held-out phase-only baseline", Y_eval, phase_only_pred)


Train: MSE=28.076088; weighted R²=0.8447; per-PC R²=[0.8665086 0.8237032 0.7586791]
Held-out: MSE=28.504120; weighted R²=0.8439; per-PC R²=[0.8660975  0.82290655 0.752505  ]
Held-out global-mean baseline: MSE=182.556503; weighted R²=-0.0001; per-PC R²=[ 7.0929527e-06 -7.5101852e-06 -5.7911873e-04]
Held-out phase-only baseline: MSE=71.623726; weighted R²=0.6076; per-PC R²=[0.8160107  0.07504755 0.47564942]


(71.62372589111328, 0.6076407432556152)

## Residual error and variance reduction as a function of phase

The first plot shows mean held-out reconstruction error by normalized phase. The second compares the model residual MSE with the raw within-phase activation variance, reporting the fraction of variance removed by the model in each phase bin.


In [15]:
eval_phase = counts_eval / widths_eval[:, None]
eval_phase = np.clip(eval_phase, 0.0, 1.5)
eval_errors = np.linalg.norm(Y_eval - Y_eval_pred, axis=-1)
phase_bins = np.linspace(0.0, 1.5, 16)
phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2
phase_error_means = []
phase_counts = []
phase_variances = []
phase_mses = []
phase_variance_reductions = []
for left, right in zip(phase_bins[:-1], phase_bins[1:]):
    mask = (eval_phase >= left) & (eval_phase < right)
    phase_counts.append(mask.sum())
    if not mask.any():
        phase_error_means.append(np.nan)
        phase_variances.append(np.nan)
        phase_mses.append(np.nan)
        phase_variance_reductions.append(np.nan)
        continue
    actual_in_bin = Y_eval[mask]
    predicted_in_bin = Y_eval_pred[mask]
    residuals_in_bin = actual_in_bin - predicted_in_bin
    phase_error_means.append(np.mean(np.linalg.norm(residuals_in_bin, axis=-1)))
    phase_mse = np.mean(residuals_in_bin ** 2)
    phase_variance = np.mean((actual_in_bin - actual_in_bin.mean(axis=0)) ** 2)
    phase_mses.append(phase_mse)
    phase_variances.append(phase_variance)
    phase_variance_reductions.append(
        1.0 - phase_mse / phase_variance if phase_variance > 0 else np.nan
    )

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=phase_centers, y=phase_error_means, mode="lines+markers",
    name="Held-out residual", customdata=np.asarray(phase_counts),
    hovertemplate="phase=%{x:.2f}<br>mean error=%{y:.4f}<br>samples=%{customdata}<extra></extra>",
))
fig.update_layout(
    title="Held-out reconstruction error by normalized within-line phase",
    xaxis_title="Character count / line length",
    yaxis_title="Euclidean error in global 3D PCA space",
    height=600,
)
display(fig)

fig_variance = go.Figure()
fig_variance.add_trace(go.Scatter(
    x=phase_centers, y=np.asarray(phase_variance_reductions) * 100,
    mode="lines+markers", name="Variance reduction",
    customdata=np.column_stack([phase_variances, phase_mses, phase_counts]),
    hovertemplate=(
        "phase=%{x:.2f}<br>variance reduction=%{y:.2f}%<br>"
        "within-bin variance=%{customdata[0]:.6f}<br>"
        "model residual MSE=%{customdata[1]:.6f}<br>"
        "samples=%{customdata[2]}<extra></extra>"
    ),
))
fig_variance.add_hline(y=0, line_dash="dot", line_color="gray")
fig_variance.update_layout(
    title="Held-out variance reduction by normalized within-line phase",
    xaxis_title="Character count / line length",
    yaxis_title="Variance reduction (%)",
    height=600,
)
display(fig_variance)


## Random held-out example in global 3D PCA space

In [14]:
example_rng = np.random.default_rng(RANDOM_EXAMPLE_SEED)
example_index = int(example_rng.integers(N_EVAL_EXAMPLES))
actual = Y_eval[example_index]
predicted = Y_eval_pred[example_index]
counts = counts_eval[example_index]
line_length = widths_eval[example_index]

def add_origin_vectors(fig, coordinates, name, color, showlegend=False):
    segments = []
    for point in coordinates:
        segments.extend([[0, 0, 0], point.tolist(), [None, None, None]])
    fig.add_trace(go.Scatter3d(
        x=[p[0] for p in segments], y=[p[1] for p in segments], z=[p[2] for p in segments],
        mode="lines", line=dict(color=color, width=1), showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter3d(
        x=coordinates[:, 0], y=coordinates[:, 1], z=coordinates[:, 2],
        mode="lines+markers", marker=dict(size=5, color=counts, colorscale="Viridis",
        colorbar=dict(title="Character count") if showlegend else None),
        line=dict(color=color, width=4), name=name,
        text=[f"position {i}" for i in range(seq_len)],
        customdata=counts,
        hovertemplate="%{text}<br>character count=%{customdata:.1f}<extra>" + name + "</extra>",
    ))

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode="markers", marker=dict(size=8, color="black"),
    name="Origin", hovertemplate="(0, 0, 0)<extra>Origin</extra>",
))
add_origin_vectors(fig, actual, "Actual activation trajectory", "royalblue", True)
add_origin_vectors(fig, predicted, "Model prediction", "crimson", False)
fig.update_layout(
    title=(f"Held-out example {example_index}: actual vs. shared phase model<br>"
           f"line length={line_length:.0f}; global PCA variance explained={pca.explained_variance_ratio_.sum():.1%}"),
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    height=850, margin=dict(l=0, r=0, b=0, t=100),
)
display(fig)
print(f"Random held-out example index: {example_index}")


Random held-out example index: 354
